## 知识库的构建
### 文档解析-->分块-->embedding-->向量数据库

In [4]:
from model import RagEmbedding, RagLLM
from doc_parse import chunk, read_and_process_excel, logger

In [1]:
from model import RagEmbedding, RagLLM
from doc_parse import chunk, read_and_process_excel, logger

In [2]:
import pandas as pd
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

In [3]:
pdf_files = ['./data/zhidu_employee.pdf',
            './data/zhidu_travel.pdf']

excel_files = ['./data/zhidu_detail.xlsx']

In [4]:
r_spliter = RecursiveCharacterTextSplitter(
    chunk_size=128,
    chunk_overlap=30,
    separators=[
        "\n\n",
        "\n",
        ".",    # 英文句号
        "．",   # 全角句号 (对应 \uff0e)
        "。",   # 中文句号 (对应 \u3002)
        ",",    # 英文逗号
        "，",   # 中文逗号 (对应 \uff0c)
        "、",   # 中文顿号 (对应 \u3001)
    ]
)

In [5]:
doc_data = []
for pdf_file_name in pdf_files:
    res = chunk(pdf_file_name, callback=logger)
    for data in res:
        content = data["content_with_weight"]
        if '<table>' not in content and len(content) > 200:
            doc_data = doc_data + r_spliter.split_text(content)
        else:
            doc_data.append(content)

OCR is running...


OCR finished.
OCR: 1.2111687500000698
preprocess
Layout analysis finished.
layouts: 1.4407863330002328
preprocess
Table analysis finished.
Text merging finished


OCR is running...
OCR finished.
OCR: 0.5604025410002578
preprocess
Layout analysis finished.
layouts: 0.7524090000001706
preprocess
Table analysis finished.
Text merging finished


In [6]:
for i in doc_data:
    print(len(i), "="*10,  i)

530 ========== <table><caption>病假发放标准：</caption>
<tr><td  >病假时间</td><td  >连续工龄</td><td  >发放标准</td></tr>
<tr><td></td><td  >不满二年</td><td  >60% </td></tr>
<tr><td></td><td  >已满二年不满四年</td><td  >70% </td></tr>
<tr><td  >6 个月以内病假</td><td  >已满四年不满六年</td><td  >80% </td></tr>
<tr><td></td><td  >已满六年不满八年</td><td  >90% </td></tr>
<tr><td></td><td  >八年以上</td><td  >100% </td></tr>
<tr><td></td><td  >不满一年</td><td  >40% </td></tr>
<tr><td  >6 个月以上病假</td><td  >已满一年不满三年</td><td  >50% </td></tr>
<tr><td></td><td  >连续工龄三年以上</td><td  >60% </td></tr>
</table>
58 ========== 教职工考勤管理制度
第一节适用范围
1、本制度包括了考勤、休假、加班等方面的规定。2、本制度适用于学校全体教职员工。
7 ========== 第二节考勤规定
66 ========== 1、学校的工作时间由学校决定并公布。学校内除特殊岗位特别规定外，全体教职员工均应严格执行学校的作息时间表，不迟到、不早退、不中途离校
120 ========== 。工作时间：星期一至星期四7:55-16:55 星期五7:55-16:152、所有教职工实行考勤打卡制度，工作日内，每天需打卡两次，上午上班一次和下午下班一次。3、教职员工因故(特殊情况除外)晚到或早退，应事先履行请假手续，经批准后方可离校
59 ========== 。原则上，员工请假无论时间长短、假期形式，除急诊病假或突发事件外，一律需按照请假流程。请假，需事先在钉钉系统中提交申请
102 ========== 。请假，需事先在钉钉系统中提交申请。有效的请假流程为：（1）员工休假必须事先向部门负责人申请，将工作交接清楚

In [7]:
for excel_file_name in excel_files:
    data = read_and_process_excel(excel_file_name)
    df = pd.DataFrame(data[8:], columns=data[7])
    data_excel = df.drop(columns=df.columns[11:17])
    content = data_excel.to_markdown(index=False).replace(' ', "")
    doc_data.append(f"### 以下是中央和国家机关工作人员赴地方差旅住宿费标准明细表： \n\n {content}")

In [8]:
from langchain_core.documents import Document
import uuid
documents = []
document_ids = []
for idx, chunk in enumerate(doc_data):
    is_table = 0
    if "table" in chunk:
        is_table = 1
    if idx == len(doc_data) - 1:
        is_table = 1
    doc_id = str(uuid.uuid4())
    document = Document(
        page_content=chunk,
        metadata={"type": "ori", "is_table": is_table})
    documents.append(document)
    document_ids.append(doc_id)

In [11]:
%pip install dashscope


  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 1.6 MB/s  0:00:01 eta 0:00:01
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dashscope]/2 [dashscope]
Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
from langchain_community.embeddings import DashScopeEmbeddings

os.environ["DASHSCOPE_API_KEY"] = "sk-ws-H.PHPPRYP.LbX3.MEYCIQDIz-wrCcL4dQUTaV-L3TSXK2Dld6oqLJn5SO8QKWfRKAIhANQIGIFNNXJZnRqUUkYNEonmfJRLZvhkg17LY8cvkLeu"

embedding = DashScopeEmbeddings(
    model="qwen3.7-text-embedding"
)

In [17]:
# embedding_cls = RagEmbedding()
from langchain_community.embeddings import DashScopeEmbeddings

embedding = DashScopeEmbeddings(
    model="qwen3.7-text-embedding"
)

In [18]:
doc_txts = dict(zip(document_ids, documents))

In [19]:
import pickle
with open('./data/zhidu_db.pickl', 'wb') as file:
    pickle.dump(doc_txts, file)

In [25]:
from chromadb.config import Settings
chroma_client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    settings=Settings(anonymized_telemetry=False),
)

In [38]:
# embedding_db = Chroma.from_documents(documents,
#                                      embedding.get_embedding_fun(),
#                                      Aclient=chroma_client,
#                                      collection_name="zhidu_db",
#                                      )
# 这里只连接或创建集合，不提交文档
embedding_db = Chroma(
    collection_name="zhidu_db_bailian",
    embedding_function=embedding,
    client=chroma_client,
)

# 在循环里，每次提交最多 20 个文档
for start in range(0, len(documents), 20):
    batch = documents[start:start + 20]
    batch_ids = document_ids[start:start + 20]

    embedding_db.add_documents(
        documents=batch,
        ids=batch_ids,
    )

    print(f"已写入 {start + len(batch)}/{len(documents)} 个文档")

已写入 20/42 个文档
已写入 40/42 个文档
已写入 42/42 个文档


In [39]:
query = '迟到有什么规定？'

In [29]:
related_docs = embedding_db.similarity_search(query, k=2)

In [30]:
related_docs

[Document(metadata={'is_table': 0, 'type': 'ori'}, page_content='1. 迟到、早退(1)劳动考勤是公司支付薪资的依据和职工年度岗位考核内容之一。（2）迟到或早退60分钟以上（含60 分钟），每次视同缺勤1 天。（3）职工迟到、早退、脱岗累计超过3 次的（含），从第1 次起，每次扣减工资50 元。\n2. 缺打卡\n员工上下班无打卡或者有效签注，又无有效证明的视为旷工或早退半天。'),
 Document(metadata={'is_table': 0, 'type': 'ori'}, page_content='。如遇紧急情况，口头申请请假的，应在上班后两天内办理补请假手续，未在规定时间内办理的，逾期无效，按旷工处理。4、无工作理由，超过上班时间到岗的，视为迟到；未到下班时间提前离校的，视为早退；中途未经批准离校，视为旷工；迟到、早退、旷工者按照相关办法处理')]

In [31]:

chroma_client = chromadb.HttpClient(host="localhost", port=8000)

def delete_chroma_collection(name, embedding_fn, chroma_client):
    collection = Chroma(name,
                        embedding_fn,
                        client=chroma_client)
    Chroma.delete_collection(collection)

In [33]:
delete_chroma_collection("zhidu_db_bailian", embedding, chroma_client)

## RAG问答流程

In [34]:
llm = RagLLM()

In [35]:
prompt_template = """
你是企业员工助手，熟悉公司考勤和报销标准等规章制度，需要根据提供的上下文信息context来回答员工的提问。\
请直接回答问题，如果上下文信息context没有和问题相关的信息，请直接回答[不知道,请咨询HR] \
问题：{question} 
"{context}"
回答：
"""

In [46]:
def run_rag_pipline(querys, k=3):
    for query in querys:
        
        related_docs = embedding_db.similarity_search(query, k=k)
        context = "\n".join([f"上下文{i+1}: {doc.page_content} \n" \
                             for i, doc in enumerate(related_docs)])
        print()
        print()
        print("#"*100)
        print(f"query: {query}")
        print(f"context: {context}")
        llm_prompt = prompt_template.replace("{question}", query).replace("{context}", context)
        response = llm(llm_prompt, stream=True)
        print(f"response: ")
        for chunk in response:
            if chunk.choices:
                print(chunk.choices[0].delta.content or "", end="", flush=True)
        # for chunk in response:
        #     print(chunk.choices[0].text, end='', flush=True)

In [43]:
import importlib
import model

print("实际导入的文件：", model.__file__)

# 重新加载已保存的 model.py
importlib.reload(model)

# 使用更新后的类，重新创建对象
llm = model.RagLLM()

print("当前接口：", llm.client.base_url)

实际导入的文件： /Users/dengyixuan/learnLargeModel/Projects/pythonProject/rag_full_stack_course_notebooks/notebook/model.py
当前接口： https://api.deepseek.com


In [47]:
run_rag_pipline(['加班有加班费吗？'])



####################################################################################################
query: 加班有加班费吗？
context: 上下文1: 。（加班需提前申请），加班需有打卡记录，无打卡记录支撑的加班时间，不计加班。加班费按照实际加班时长的2倍计算。3、加班以调休等额返还（代替），凡调休人员应填写《请假申请表》，选择“调休”一栏，经所在部门分管领导签字后，交由人事处核实备案 

上下文2: 1、学校以下列日期为例行公休日(若有变更需事先公布)，但因学校工作需要可指定照常上班，以加班计算：(1)法定节假日(2)周六、周日2、正常工作日，延长工作时间连续达到4 小时以上的方可计算加班 

上下文3: 。8、调休：各职能部门因工作需要，需要在工作日、节假日安排本部门加班或值班的，应由行政部门书面报送至人力资源部，如遇突发情况可事后补办手续。审批通过后，加班或值班时间可申请调休，教职工本人需填写《员工请假申请单》，报相关部门人员签字，并交给人事部核算考勤 

response: 
根据公司规定，加班有加班费，按照实际加班时长的2倍计算。但需注意：加班需提前申请，且有打卡记录支撑，无打卡记录不计加班。另外，加班也可以调休等额返还（代替加班费）。

In [48]:
run_rag_pipline(['出差可以买意外保险吗？需要自己购买吗'])



####################################################################################################
query: 出差可以买意外保险吗？需要自己购买吗
context: 上下文1: 差旅费用标准
差旅费开支范围包括工作人员临时到常驻地以外地区公务出差所发生的城市间交通费、住宿费、伙食补助费和市内交通费。一、城市间交通费城市间交通费是指工作人员因公到常驻地以外地区出差乘坐火车、轮船、飞机等交通工具所发生的费用。1.出差人员在不影响公务、确保安全的前提下，选乘经济便捷的交通工具。2.出差人员要按照规定等级乘坐交通工具，未按规定乘坐的，超支部分自理。乘坐交通工具的等级见下表：
<table>
<tr><td  >级别</td><td  >火车 （含高铁、动车、全列软席列车）</td><td  >轮船 （不包括 旅游船）</td><td  >飞机</td><td  >其他交通工具 （不包括出租 小汽车）</td></tr>
<tr><td  >享受副部级 待遇及以上 人员</td><td  >火车软席（软座、软卧），高铁/动车商 务座，全列软席列车一等软座</td><td  >一等舱</td><td  >头等舱</td><td  >凭据报销</td></tr>
<tr><td  >秘书长及副 秘书长</td><td  >火车软席（软座、软卧），高铁/动车一 等座，全列软席列车一等软座</td><td  >二等舱</td><td  >经济舱</td><td  >凭据报销</td></tr>
<tr><td  >其余人员</td><td  >火车硬席（硬座、硬卧），高铁/动车二 等座、全列软席列车二等软座</td><td  >三等舱</td><td  >经济舱</td><td  >凭据报销</td></tr>
</table>
备注：
享受副部级待遇及以上人员出差，因工作需要，随行一人可乘坐同等级交通工具；乘坐飞机的，民航发展基金、燃油附加费可以凭据报销；乘坐飞机、火车、轮船等交通工具的，每人次可以购买交通意外保险一份。由我会统一购买交通意外保险的，不再重复购买。3.我会工作人员出差，高铁最短5 个小时内能够到达目的地的，原则上应乘坐高铁，如遇特殊情况，经分管副秘书长、秘书

In [ ]:
run_rag_pipline(['那个，我们公司有什么规定来着？ 您公司的具体规定需要参考您所在公司的员工手册或相关政策文件。通常，公司会有包括但不限于工作时间、请假流程、加班政策、行为准则、保密协议等方面的规章制度。建议查阅您的员工手册或向人力资源部门咨询以获取详细信息。'])

In [ ]:
run_rag_pipline(['公司派我去上海，钱怎么弄？'])

In [ ]:
run_rag_pipline(['如果我明天要出去，那个报销的事情怎么办？'])

In [ ]:
run_rag_pipline(['如果我明天要出去，那个报销的事情怎么办？ 如果您明天需要外出，关于报销的事宜，请提前与您的直接上级或财务部门沟通。确保了解公司的报销流程和所需文件，如需预支费用，应按照公司规定提交申请。同时，保留好所有相关票据，以便返回后及时完成报销手续。如果可能，可以预先填写报销单的部分内容，以加快后续处理速度。'])